# Prompt Template
- PromptTemplate
- ChatPromptTemplate(推荐)

In [4]:
from idlelib import history

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pathlib import Path

# 从.env文件中加载环境变量
load_dotenv(Path('../.env'), override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model='qwen-max',
    model_provider='openai',
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL
)


## ChatPromptTemplate基础使用

In [1]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate(
    [
        ("system", "You are a helpful AI bot. Your name is {name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "{user_input}"),
    ]
)

prompt_value = template.invoke({
    "name": "Peter",
    "user_input": "1+1=?"
})

print(prompt_value)

messages=[SystemMessage(content='You are a helpful AI bot. Your name is Peter.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1+1=?', additional_kwargs={}, response_metadata={})]


## ChatPromptTemplate高级特性
### partial()预填充
适合填充公共变量

In [6]:
prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '你是一个智能客服助手, 名字是{name}.'),
        ("human", "{user_input}"),
    ]
)

partial_prompt = prompt.partial(name='小智')

prompt_value = partial_prompt.invoke(
    {"user_input": "你叫什么名字？"}
)

resp = model.invoke(prompt_value)
print(resp)

content='您好，我叫小智，是您的智能客服助手。有什么可以帮助您的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 30, 'total_tokens': 48, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-c57bdafc-f9e8-9db9-a38a-fb21cfc005c3', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f6fb5-c445-76e0-a5a4-5ad37a5b69b7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 30, 'output_tokens': 18, 'total_tokens': 48, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


### 消息占位符
#### placeholder
可以是一个消息列表

In [8]:
prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '你是一个智能客服助手, 名字是{name}.'),
        ('placeholder', "{history}"),
        ("human", "{user_input}"),
    ]
)

partial_prompt = prompt.partial(name='小智')

prompt_value = partial_prompt.invoke(
    {
        "user_input": "我叫什么名字?我是干什么的？",
        "history": [
            ('human', "你叫什么名字？我叫Peter，我是一个程序员"),
            ('ai', "Peter, Hello! 我叫小智！")
        ],
    }
)

print(f'prompt value: {prompt_value}')
print('-' * 50)

resp = model.invoke(prompt_value)
print(resp)

prompt value: messages=[SystemMessage(content='你是一个智能客服助手, 名字是小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你叫什么名字？我叫Peter，我是一个程序员', additional_kwargs={}, response_metadata={}), AIMessage(content='Peter, Hello! 我叫小智！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我叫什么名字?我是干什么的？', additional_kwargs={}, response_metadata={})]
content='你叫Peter，你是一名程序员。很高兴认识你，Peter！如果你有任何问题或需要帮助的地方，随时告诉我哦！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 66, 'total_tokens': 92, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-1c4d78ab-8636-9fb5-a322-47c519770165', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f6fbb-19b0-7c13-8755-f324b7599d13-0' tool_calls=[] invalid

#### MessagePlaceholder

In [9]:
from langchain_core.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '你是一个智能客服助手, 名字是{name}.'),
        MessagesPlaceholder(variable_name='history'),
        ("human", "{user_input}"),
    ]
)

partial_prompt = prompt.partial(name='小智')

prompt_value = partial_prompt.invoke(
    {
        "user_input": "我叫什么名字?我是干什么的？",
        "history": [
            ('human', "你叫什么名字？我叫Peter，我是一个程序员"),
            ('ai', "Peter, Hello! 我叫小智！"),
        ],
    }
)

print(f'prompt value: {prompt_value}')
print('-' * 50)

resp = model.invoke(prompt_value)
print(resp)

prompt value: messages=[SystemMessage(content='你是一个智能客服助手, 名字是小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你叫什么名字？我叫Peter，我是一个程序员', additional_kwargs={}, response_metadata={}), AIMessage(content='Peter, Hello! 我叫小智！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我叫什么名字?我是干什么的？', additional_kwargs={}, response_metadata={})]
--------------------------------------------------
content='你叫Peter，你是一名程序员。很高兴认识你，Peter！如果你有任何问题，无论是关于编程还是其他方面，我都很乐意帮忙。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 66, 'total_tokens': 96, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-996fcffb-66cb-9e54-ac37-f50ee10e5df0', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019

### 可复用模板库
把提示词抽取为公共模块,按需导入

### 组合模版

In [11]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = ChatPromptTemplate(
    [
         ('system', '你是一个智能客服助手, 名字是小智.'),
    ]
)

history_prompt = ChatPromptTemplate(
    [
        ('human', "你叫什么名字？我叫Peter，我是一个程序员"),
        ('ai', "Peter, Hello! 我叫小智！"),
    ]
)

user_prompt = ChatPromptTemplate(
    [
        ('user', '我叫什么名字?我是干什么的？')
    ]
)

prompt_value = system_prompt + history_prompt + user_prompt

print(f'prompt value: {prompt_value}')

prompt value: input_variables=[] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='你是一个智能客服助手, 名字是小智.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='你叫什么名字？我叫Peter，我是一个程序员'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Peter, Hello! 我叫小智！'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='我叫什么名字?我是干什么的？'), additional_kwargs={})]
